# Eksplorasi AutoModelForTokenClassification

**Task**: Sequence Labeling (Named Entity Recognition [NER], Part-of-Speech [POS] Tagging, dll.)
**Cara Kerja**: Menerima suatu teks, lalu memberikan label (klasifikasi) pada setiap kata atau token (subword) di dalam teks tersebut.
**Model Populer**: BERT-NER, RoBERTa, DeBERTa, dll.
**Dataset**: `conll2003` - dataset standar dan paling populer untuk melatih model Named Entity Recognition (NER). Dataset ini berisi berita dengan tag entitas seperti orang (PER), lokasi (LOC), organisasi (ORG), dan lainnya.

In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from datasets import load_dataset
import torch

## 1. Load Dataset Publik (`conll2003`)
Kita muat dataset CoNLL-2003. Dalam dataset ini, setiap kalimat sudah dipisah menjadi kata-kata (tokens), dan setiap kata memiliki label (ner_tags). Karena bentuknya berupa deretan kata, task ini sering juga disebut *Sequence Labeling*.

In [2]:
dataset = load_dataset("lhoestq/conll2003", split="train")

# Melihat daftar label yang tersedia
label_list = dataset.features["ner_tags"].feature.names
print("Daftar Label NER:\n", label_list)

print("\nContoh data index ke-0:")
tokens = dataset[0]["tokens"]
tags = dataset[0]["ner_tags"]

for token, tag_id in zip(tokens, tags):
    print(f"{token:15} -> {label_list[tag_id]}")

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Daftar Label NER:
 ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

Contoh data index ke-0:
EU              -> B-ORG
rejects         -> O
German          -> B-MISC
call            -> O
to              -> O
boycott         -> O
British         -> B-MISC
lamb            -> O
.               -> O


In [3]:
for data in dataset:
    print(data)
    break

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


## 2. Load Tokenizer & Model
Kita akan menggunakan model `dslim/bert-base-NER`, yaitu model BERT base yang sudah di fine-tuning khusus pada dataset NER (seperti conll2003) agar bisa langsung memprediksi entitas.

In [4]:
model_checkpoint = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Pastikan menggunakan AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
print(model)

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

## 4. Persiapan Data untuk PyTorch Training Mandiri
Pada kasus Token Classification (Named Entity Recognition), **setiap token** memerlukan sebuah label. Ketika model tokenization memecah sebuah kata (misal "Indonesian" menjadi "Indo", "nes", "ian"), kita wajib memperluas dan mencocokkan *subwords* tersebut dengan label NER kata aslinya. PyTorch membutuhkan ini dalam format Tensor.

In [6]:
from torch.utils.data import Dataset, DataLoader
import torch

train_sample = dataset.select(range(50)) # 50 baris untuk contoh

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=128, padding="max_length"
    )
    labels = []
    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Mengembalikan indeks kata masing-masing token
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens (seperti [CLS], [SEP], [PAD]) memiliki id None, atur label ke -100 agar diabaikan oleh Loss Function
            if word_idx is None:
                label_ids.append(-100)
            # Berikan label pertama dari kata aslinya ke token subword yang pertama muncul
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Untuk token subword kelanjutan (contoh, ##nesian dari Indonesian), abaikan di Loss function (atau set ke id label yg sama, di sini kita coba -100).
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_sample.map(tokenize_and_align_labels, batched=True, remove_columns=dataset.column_names)

class TokenClassificationDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

train_dataloader = DataLoader(TokenClassificationDataset(tokenized_train), batch_size=4, shuffle=True)
print(f"Total Batch Data Token Classification: {len(train_dataloader)}")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Total Batch Data Token Classification: 13


In [9]:
for data in train_dataloader:
    print(data['labels'])
    break

tensor([[-100,    0, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100],
        [-100,    0, -100,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    3, 

## 5. Proses PyTorch Training Loop
Melatih model token classification berarti melakukan perhitungan loss pada *setiap* token sequence secara individual, kecuali pada token dengan index label "-100" (otomatis di-*masking* oleh `CrossEntropyLoss`).

In [10]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training Token Classification (NER) ===")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device) # Shape [Batch, Seq_Length]
        
        # Logits berupa [Batch Size, Sequence Length, Num_Labels]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

==== Memulai Training Token Classification (NER) ===
Epoch 1 | Step 0 | Loss 1.5382
Epoch 1 | Step 5 | Loss 0.4777
Epoch 1 | Step 10 | Loss 0.1781
>> Rata-rata Train Loss Epoch 1: 0.6328



## 6. Evaluasi dan Pengujian Inference
Untuk pengenalan entitas unik (NER), kita mengevaluasi prediksi label setiap token. Untuk melihat contoh prediksinya, mari kita jalankan teks baru ke arsitektur jaringan PyTorch.

In [11]:
test_ner_sentence = "My name is John Doe and I work at HuggingFace in New York."

model.eval()
input_tensors = tokenizer(test_ner_sentence, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**input_tensors)

# Mengambil prediksi terkuat di dimensi terakhir untuk setiap token
predictions = outputs.logits.argmax(dim=-1).squeeze().tolist()
tokens = tokenizer.convert_ids_to_tokens(input_tensors["input_ids"].squeeze())

print("==== Hasil Prediksi Named Entity Recognition ====")
for token, pred_idx in zip(tokens, predictions):
    # Menyaring token khusus sub-words dan Padding
    if token not in [tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token]:
        entity_name = model.config.id2label[pred_idx]
        print(f"{token.ljust(15)} : {entity_name}")

==== Hasil Prediksi Named Entity Recognition ====
My              : O
name            : O
is              : O
John            : B-PER
Do              : I-PER
##e             : O
and             : O
I               : O
work            : O
at              : O
Hu              : B-ORG
##gging         : O
##F             : I-ORG
##ace           : O
in              : O
New             : B-ORG
York            : I-LOC
.               : O
